# Solution to p06

Reshaping separates the height and width into patch-grid and within-patch axes. The permutation puts the two grid axes before channel and within-patch axes; the final reshape then flattens each patch channel-first while preserving row-major grid order.

In [ ]:
import torch

images = torch.tensor([
    [[[0., 1., 2., 3.], [4., 5., 6., 7.], [8., 9., 10., 11.], [12., 13., 14., 15.]]],
    [[[16., 17., 18., 19.], [20., 21., 22., 23.], [24., 25., 26., 27.], [28., 29., 30., 31.]]],
], dtype=torch.float32)
patch_side = 2
ATOL = 0.0
RTOL = 0.0

def patchify(images, patch_side):
    if not isinstance(images, torch.Tensor) or images.ndim != 4:
        raise ValueError("images must be a 4-D tensor")
    if images.dtype != torch.float32 or images.device.type != "cpu":
        raise ValueError("images must be CPU float32")
    if not isinstance(patch_side, int) or patch_side <= 0:
        raise ValueError("patch_side must be a positive integer")
    B, C, H, W = images.shape
    if H % patch_side or W % patch_side:
        raise ValueError("height and width must be divisible by patch_side")
    grid_h, grid_w = H // patch_side, W // patch_side
    separated = images.reshape(B, C, grid_h, patch_side, grid_w, patch_side)
    ordered = separated.permute(0, 2, 4, 1, 3, 5).contiguous()
    return ordered.reshape(B, grid_h * grid_w, C * patch_side * patch_side)

tokens = patchify(images, patch_side)


### Answer check

In [ ]:
expected = torch.tensor([
    [[0,1,4,5],[2,3,6,7],[8,9,12,13],[10,11,14,15]],
    [[16,17,20,21],[18,19,22,23],[24,25,28,29],[26,27,30,31]],
], dtype=torch.float32)
assert tokens.shape == (2,4,4) and tokens.dtype == torch.float32
torch.testing.assert_close(tokens, expected, atol=ATOL, rtol=RTOL)
torch.testing.assert_close(tokens[0,0], torch.tensor([0.,1.,4.,5.]), atol=ATOL, rtol=RTOL)
torch.testing.assert_close(tokens[1,3], torch.tensor([26.,27.,30.,31.]), atol=ATOL, rtol=RTOL)
for bad in (torch.zeros(2,4,4), torch.zeros(1,1,3,4)):
    try:
        patchify(bad, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input was not rejected")
